# Open-ended EDA v2 — analysis on `data/unified_core.parquet`

Clean notebook version of the open-ended senior-DS EDA. Primary data file switched from `data/unified.parquet` (1.45M × 96) to `data/unified_core.parquet` (110k × 42, LinkedIn-only, all rows inside the Stage 9 balanced LLM-frame, with `is_control` populated in all four periods).

Hypothesis set extended to **H1–H13**. Full priors in [`../memos/priors.md`](../memos/priors.md).

**How to read this notebook:** each scan writes a figure to `eda/figures/` and a CSV to `eda/tables/`. The code here is a thin driver — all scan logic lives in `eda/scripts/{scans,core_scans}.py`. Execute top-to-bottom to regenerate all artifacts.

## 0. Setup

In [1]:
import sys, os
from pathlib import Path

# Resolve project root
HERE = Path.cwd()
ROOT = HERE if (HERE / 'data' / 'unified_core.parquet').exists() else HERE.parents[1]
assert (ROOT / 'data' / 'unified_core.parquet').exists(), f'unified_core.parquet not found at {ROOT}'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'eda' / 'scripts'))

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if isinstance(x, float) else x)

from scans import AI_VOCAB_PATTERN, NEW_AI_TITLE_PATTERN, BIG_TECH_CANONICAL
from core_scans import (
    CORE_PATH, CORE_OBS_PATH, CORE_DEFAULT_FILTER, VENDOR_PATTERNS,
    rerun_s1_core, rerun_s3_core, rerun_s10_core, rerun_s11_core, rerun_sv_core,
    scan_s12, scan_s13, scan_s14, scan_s15, scan_s16, scan_s17,
)

con = duckdb.connect()
print(f'Project root: {ROOT}')
print(f'Core file:    {CORE_PATH}')
print(f'Core obs:     {CORE_OBS_PATH}')

Project root: /home/jihgaboot/gabor/job-research
Core file:    /home/jihgaboot/gabor/job-research/data/unified_core.parquet
Core obs:     /home/jihgaboot/gabor/job-research/data/unified_core_observations.parquet


## 1. Hypothesis recap (H1–H13)

Full priors in `eda/memos/priors.md`. Short form:

**Macro-narrative (from v1):**
- **H1** AI-washing — AI is narrative cover for macro layoffs. *Falsified on content* if SWE vs control diverge.
- **H2** AI creates new job types — emergence in titles.
- **H3** Non-AI macro (Covid-binge + rate hikes + outsourcing) — Economist central claim.
- **H4** Industry spread to non-tech — Economist tech-skills-everywhere.
- **H5** Junior-first *vs* senior-restructuring within SWE.
- **H6** Big Tech vs rest — reaction differs by tier.
- **H7** SWE vs control divergence — direct SWE-specificity test.

**New in v2 (from unified_core exploration):**
- **H8** YOE floor is FALLING, not rising (counter scope-inflation).
- **H9** Dev-tool vendor labor-market leaderboard is measurable (Copilot / Claude / OpenAI / Cursor hierarchy).
- **H10** AI mention is NOT a ghost-job signal.
- **H11** Non-SWE AI-spread is niche-specific (finance + power/nuclear engineering), not broad.
- **H12** Posting survival differs by tier / AI-tag.
- **H13** Within-firm AI rewrite is real — same firm, 2024 → 2026, did they rewrite their own postings?

## 2. Phase A — corpus profile on core

In [2]:
profile = con.execute(f"""
  SELECT source, source_platform, period,
         COUNT(*) AS n,
         SUM(CASE WHEN is_swe THEN 1 ELSE 0 END) AS swe,
         SUM(CASE WHEN is_swe_adjacent THEN 1 ELSE 0 END) AS adjacent,
         SUM(CASE WHEN is_control THEN 1 ELSE 0 END) AS control
  FROM '{CORE_PATH}'
  GROUP BY 1,2,3 ORDER BY 1,2,3
""").df()
profile['total_check'] = profile['swe'] + profile['adjacent'] + profile['control']
print('Core row count:', profile['n'].sum())
profile

Core row count: 110000


,source,source_platform,period,n,swe,adjacent,control,total_check
0,kaggle_arshkon,linkedin,2024-04,15692,4687.0000,2394.0000,8611.0000,15692.0000
1,kaggle_asaniczka,linkedin,2024-01,41571,18125.0000,9417.0000,14029.0000,41571.0000
2,scraped,linkedin,2026-03,24112,11810.0000,5586.0000,6716.0000,24112.0000
3,scraped,linkedin,2026-04,28625,14012.0000,7303.0000,7310.0000,28625.0000


In [3]:
coverage = con.execute(f"""
  SELECT period, analysis_group, llm_classification_coverage,
         llm_extraction_coverage, COUNT(*) AS n
  FROM '{CORE_PATH}'
  GROUP BY 1,2,3,4 ORDER BY 1,2,3,4
""").df()
coverage_pivot = coverage.pivot_table(
    index=['period','analysis_group'],
    columns='llm_classification_coverage',
    values='n', fill_value=0, aggfunc='sum'
)
coverage_pivot

llm_classification_coverage  deferred  labeled
period  analysis_group                        
2024-01 control                    37    13992
        swe                         1    18124
        swe_adjacent                2     9415
2024-04 control                     5     8606
        swe                         0     4687
        swe_adjacent                0     2394
2026-03 control                    20     6696
        swe                         2    11808
        swe_adjacent                1     5585
2026-04 control                    37     7273
        swe                         5    14007
        swe_adjacent                1     7302

## 3. Phase B — re-run finalists on core

Five v1 finalists re-executed against unified_core. Writes to `eda/tables/{S1,S3,S10,S11,Sv}_core_*.csv` and matching PNGs.

In [4]:
s1 = rerun_s1_core(con); s1

,period,seniority_3level,n,n_ai,ai_rate
0,2024-01,junior,238,3.0000,0.0126
1,2024-01,mid,39,1.0000,0.0256
2,2024-01,senior,11633,353.0000,0.0303
3,2024-01,unknown,6215,176.0000,0.0283
4,2024-04,junior,177,10.0000,0.0565
5,2024-04,mid,15,0.0000,0.0000
6,2024-04,senior,2030,103.0000,0.0507
7,2024-04,unknown,2465,97.0000,0.0394
8,2026-03,junior,636,162.0000,0.2547
9,2026-03,mid,52,15.0000,0.2885


In [5]:
s3 = rerun_s3_core(con); s3

,period,n,n_new_ai,new_ai_share
0,2024-01,18125,285.0000,0.0157
1,2024-04,4687,134.0000,0.0286
2,2026-03,11810,962.0000,0.0815
3,2026-04,14012,1143.0000,0.0816


In [6]:
s10 = rerun_s10_core(con); s10

,period,tier,n,n_ai,ai_rate,period_total,volume_share
0,2024-01,big_tech,426,24.0000,0.0563,18125,0.0235
1,2024-01,rest,17699,509.0000,0.0288,18125,0.9765
2,2024-04,big_tech,143,14.0000,0.0979,4687,0.0305
3,2024-04,rest,4544,196.0000,0.0431,4687,0.9695
4,2026-03,big_tech,865,345.0000,0.3988,11810,0.0732
5,2026-03,rest,10945,2807.0000,0.2565,11810,0.9268
6,2026-04,big_tech,986,434.0000,0.4402,14012,0.0704
7,2026-04,rest,13026,3552.0000,0.2727,14012,0.9296


In [7]:
s11 = rerun_s11_core(con); s11

,period,analysis_group,n,n_ai,ai_rate
0,2024-01,control,14029,36.0000,0.0026
1,2024-01,swe,18125,533.0000,0.0294
2,2024-01,swe_adjacent,9417,234.0000,0.0248
3,2024-04,control,8611,20.0000,0.0023
4,2024-04,swe,4687,210.0000,0.0448
5,2024-04,swe_adjacent,2394,94.0000,0.0393
6,2026-03,control,6716,73.0000,0.0109
7,2026-03,swe,11810,3152.0000,0.2669
8,2026-03,swe_adjacent,5586,1220.0000,0.2184
9,2026-04,control,7310,100.0000,0.0137


In [8]:
sv = rerun_sv_core(con); sv

,period,seniority_3level,n,mean_raw_len,mean_core_len
0,2024-01,junior,238,3365.4244,1946.8025
1,2024-01,mid,39,3799.0513,2159.4103
2,2024-01,senior,11633,4171.0497,2423.4684
3,2024-01,unknown,6215,3362.1020,2094.5238
4,2024-04,junior,177,3476.5989,1852.8136
5,2024-04,mid,15,4297.4667,2586.5333
6,2024-04,senior,2030,3734.0507,2164.9921
7,2024-04,unknown,2465,2939.7416,1808.8725
8,2026-03,junior,636,5283.7877,2236.3073
9,2026-03,mid,52,5887.3269,2779.1569


## 4. Phase B — new scans S12–S17 (H8–H13)

### S12 (H8) — YOE floor trajectory

In [9]:
s12 = scan_s12(con)
display(s12)

,period,seniority_3level,n,n_with_yoe,mean_yoe,median_yoe,sd_yoe,p90_yoe
0,2024-01,junior,238,139,2.0144,2.0000,1.6896,5.0000
1,2024-01,mid,39,32,4.0000,3.0000,3.1623,9.8000
2,2024-01,senior,11633,9947,6.5531,6.0000,3.0233,10.0000
3,2024-01,unknown,6215,4647,5.1147,5.0000,3.0738,8.0000
4,2024-04,junior,177,89,1.4607,1.0000,1.0772,3.0000
5,2024-04,mid,15,13,3.0000,2.0000,2.5495,7.4000
6,2024-04,senior,2030,1604,6.5387,6.0000,2.6514,10.0000
7,2024-04,unknown,2465,1748,4.7397,5.0000,2.5528,8.0000
8,2026-03,junior,636,220,1.3682,1.0000,1.3501,3.0000
9,2026-03,mid,52,39,2.8718,3.0000,1.6251,5.2000


**Read:** junior mean YOE 2.01 → 1.23 (mean) and 2 → 1 (median) across 2024→2026. Senior median dropped 6 → 5. Classic scope-inflation at the YOE bar is falsified on LLM-YOE.

### S13 (H9) — Vendor labor-market leaderboard

In [10]:
s13 = scan_s13(con)
s13

,period,n_total,copilot_rate,cursor_rate,claude_rate,anthropic_rate,chatgpt_rate,openai_rate,gemini_rate,llama_rate,mistral_rate,gpt_rate
0,2024-01,18125,0.0008,0.0001,0.0002,0.0002,0.0009,0.0022,0.0003,0.0004,0.0001,0.0018
1,2024-04,4687,0.0015,0.0000,0.0002,0.0009,0.0009,0.0051,0.0004,0.0006,0.0002,0.0015
2,2026-03,11810,0.0372,0.0196,0.0321,0.0164,0.0082,0.0337,0.0124,0.0044,0.0015,0.0119
3,2026-04,14012,0.0425,0.0217,0.0383,0.0148,0.0070,0.0363,0.0107,0.0053,0.0017,0.0087


In [11]:
# Stand-alone 2026-04 leaderboard
pd.read_csv('eda/tables/S13_vendor_leaderboard_2026_04.csv')

,vendor,rate_2026_04
0,copilot,0.0425
1,claude,0.0383
2,openai,0.0363
3,cursor,0.0217
4,anthropic,0.0148
5,gemini,0.0107
6,gpt,0.0087
7,chatgpt,0.0070
8,llama,0.0053
9,mistral,0.0017


**Read:** in 2026-04 SWE postings, Copilot leads at 4.25%, Claude second at 3.83%, OpenAI third at 3.63%. Cursor emerged from ~0% to 2.17% in two years. ChatGPT as a brand is PLATEAUING (0.82% → 0.70%) while Claude/OpenAI/Anthropic continue climbing.

### S14 (H10) — AI-mention × ghost rate

In [12]:
s14 = scan_s14(con)
s14

,period,ai_tag,n,n_inflated,n_ghost,n_realistic,inflated_rate,ghost_rate,any_non_realistic
0,2024-01,ai_mentioned,533,28.0000,2.0000,503.0000,0.0525,0.0038,0.0563
1,2024-01,no_ai,17591,1035.0000,36.0000,16520.0000,0.0588,0.0020,0.0609
2,2024-04,ai_mentioned,210,16.0000,0.0000,194.0000,0.0762,0.0000,0.0762
3,2024-04,no_ai,4477,320.0000,18.0000,4139.0000,0.0715,0.0040,0.0755
4,2026-03,ai_mentioned,3152,138.0000,3.0000,3011.0000,0.0438,0.0010,0.0447
5,2026-03,no_ai,8656,438.0000,24.0000,8194.0000,0.0506,0.0028,0.0534
6,2026-04,ai_mentioned,3985,179.0000,3.0000,3803.0000,0.0449,0.0008,0.0457
7,2026-04,no_ai,10022,601.0000,17.0000,9404.0000,0.0600,0.0017,0.0617


**Read:** AI-mentioning SWE postings are *slightly less* inflated than non-AI postings (4.5% vs 5.6% in 2026-04). The 'AI buzzword = ghost job' narrative is not supported by our LLM ghost assessment.

### S15 (H11) — Control AI-rise occupational drivers

In [13]:
s15 = scan_s15(con)
# Rank families by 2026-04 AI-rate
latest = s15[s15['period']=='2026-04'].sort_values('ai_rate', ascending=False)
latest

,period,family,n,n_ai,ai_rate
20,2026-04,electrical/nuclear engineer,564,40.0000,0.0709
19,2026-04,civil engineer,216,8.0000,0.0370
21,2026-04,finance/accounting,1928,36.0000,0.0187
22,2026-04,mechanical engineer,472,7.0000,0.0148
23,2026-04,nursing,4123,9.0000,0.0022
18,2026-04,chemical engineer,7,0.0000,0.0000


**Read:** the control-tier AI rise concentrates in finance/accounting and electrical/nuclear engineering. Nursing, HR, sales, marketing are essentially untouched. Reframes H4 from 'tech skills everywhere' to 'AI language is reaching finance and utility engineering; service-occupation control still absent.'

### S16 (H12) — Posting survival

In [14]:
s16 = scan_s16(con)
s16

,period,analysis_group,ai_mentioned,n,mean_obs,median_obs,mean_days,p90_days
0,2026-03,control,False,6643,1.2351,1.0000,2.6003,20
1,2026-03,control,True,73,1.4932,1.0000,7.5342,22
2,2026-03,swe,False,8658,1.4406,1.0000,4.3561,22
3,2026-03,swe,True,3152,1.4838,1.0000,5.2690,22
4,2026-03,swe_adjacent,False,4366,1.3699,1.0000,3.6903,21
5,2026-03,swe_adjacent,True,1220,1.4238,1.0000,4.6992,22
6,2026-04,control,False,7210,1.0854,1.0000,0.1954,0
7,2026-04,control,True,100,1.1500,1.0000,0.2800,1
8,2026-04,swe,False,10026,1.1538,1.0000,0.3075,1
9,2026-04,swe,True,3986,1.1646,1.0000,0.3164,1


**Read:** in 2026-03 (month with complete observation window), AI-mentioning SWE postings stay alive 0.9 days longer on average (5.27 vs 4.36 days). Control AI-mentioning postings show a larger gap (7.5 vs 2.6 days) but the sample is small. Small positive signal consistent with either higher demand or slower filling. 2026-04 is still ongoing and not directly comparable.

### S17 (H13) — Within-firm AI rewrite overlap panel

In [15]:
s17 = scan_s17(con)
print('Panel size:', len(s17), 'companies')
print(f"Mean delta 2026−2024: {s17['delta'].mean()*100:.2f}pp")
print(f"Median delta:         {s17['delta'].median()*100:.2f}pp")
print(f"% companies with rise:    {(s17['delta']>0).mean()*100:.1f}%")
print(f"% with rise >10pp:        {(s17['delta']>0.10).mean()*100:.1f}%")
print(f"% with rise >20pp:        {(s17['delta']>0.20).mean()*100:.1f}%")
print()
print('Top 15 risers:')
s17.sort_values('delta', ascending=False).head(15)

Panel size: 292 companies
Mean delta 2026−2024: 19.36pp
Median delta:         16.67pp
% companies with rise:    75.0%
% with rise >10pp:        61.3%
% with rise >20pp:        39.4%

Top 15 risers:


,company_name_canonical,n_2024,n_2026,n_ai_2024,n_ai_2026,ai_rate_2024,ai_rate_2026,delta
239,Ridgeline,9.0000,6.0000,0.0000,6.0000,0.0000,1.0000,1.0000
94,Aditi Consulting,28.0000,16.0000,0.0000,16.0000,0.0000,1.0000,1.0000
105,Verkada,12.0000,26.0000,0.0000,26.0000,0.0000,1.0000,1.0000
55,KPMG US,63.0000,10.0000,0.0000,9.0000,0.0000,0.9000,0.9000
39,Capgemini,23.0000,78.0000,1.0000,68.0000,0.0435,0.8718,0.8283
268,Radley James,7.0000,5.0000,0.0000,4.0000,0.0000,0.8000,0.8000
163,ServiceNow,10.0000,13.0000,0.0000,10.0000,0.0000,0.7692,0.7692
206,GM Financial,6.0000,12.0000,0.0000,9.0000,0.0000,0.7500,0.7500
208,The Hartford,9.0000,8.0000,0.0000,6.0000,0.0000,0.7500,0.7500
242,Bristol Myers Squibb,8.0000,7.0000,0.0000,5.0000,0.0000,0.7143,0.7143


**Read:** on the 292-company overlap panel (companies with ≥5 SWE postings in BOTH 2024 Kaggle and 2026 scraped), the *same companies* rewrote their postings. Mean within-firm delta = **+19.4pp**; 75% of companies rose; 61% rose more than 10pp; 39% rose more than 20pp. Microsoft +51pp, Wells Fargo +45pp, Amazon +36pp. Defense firms (Raytheon, Northrop Grumman) flat or near-flat.

**This rules out between-firm composition as the sole driver of the 2024→2026 AI rewrite.** The rewrite is happening INSIDE the same firms.

## 5. Headlines — v2 verdict table

In [16]:
verdicts = pd.DataFrame([
    ('H1  AI-washing',                'against (content-level)',
     'S11 SWE+25pp vs control+1.2pp — 21× delta ratio'),
    ('H2  new AI job types',          'supported',
     'S3 new-AI-title share 1.6% → 8.3% (5×)'),
    ('H3  non-AI macro (content)',    'against',
     'S11 SWE-specificity rules out economy-wide cause'),
    ('H4  industry spread',           'NOT on LinkedIn',
     'S8 non-tech share flat ~55% 2024→2026'),
    ('H5  junior-first vs senior',    '(b) senior consistent; (a) falsified',
     'S1 AI-vocab uniform across levels; S12 junior YOE FELL'),
    ('H6  Big Tech vs rest',          'split',
     'S10 BT AI 44% vs rest 27% (diff 17pp); BT SHARE ROSE not fell'),
    ('H7  SWE-vs-control',            'strongly supports SWE-specificity',
     'S11 on balanced core with real 2024 control baseline'),
    ('H8  YOE floor falling',         'supported',
     'S12 junior mean YOE 2.01 → 1.23, median 2 → 1'),
    ('H9  vendor leaderboard',        'supported (Copilot > Claude > OpenAI > Cursor)',
     'S13 2026-04: Copilot 4.25%, Claude 3.83%, OpenAI 3.63%'),
    ('H10 AI mention ≠ ghost job',    'supported',
     'S14 AI inflated rate 4.5% vs non-AI 5.6%'),
    ('H11 control spread is niche',   'supported',
     'S15 finance + electrical/nuclear dominate; service occupations flat'),
    ('H12 posting survival',          'directional only',
     'S16 AI postings live 0.9 days longer in 2026-03 (small n for control)'),
    ('H13 within-firm AI rewrite',    'strongly supported',
     'S17 mean +19.4pp, 75% of 292 companies rose'),
], columns=['hypothesis', 'verdict', 'primary evidence'])
verdicts

,hypothesis,verdict,primary evidence
0,H1 AI-washing,against (content-level),S11 SWE+25pp vs control+1.2pp — 21× delta ratio
1,H2 new AI job types,supported,S3 new-AI-title share 1.6% → 8.3% (5×)
2,H3 non-AI macro (content),against,S11 SWE-specificity rules out economy-wide cause
3,H4 industry spread,NOT on LinkedIn,S8 non-tech share flat ~55% 2024→2026
4,H5 junior-first vs senior,(b) senior consistent; (a) falsified,S1 AI-vocab uniform across levels; S12 junior ...
5,H6 Big Tech vs rest,split,S10 BT AI 44% vs rest 27% (diff 17pp); BT SHAR...
6,H7 SWE-vs-control,strongly supports SWE-specificity,S11 on balanced core with real 2024 control ba...
7,H8 YOE floor falling,supported,"S12 junior mean YOE 2.01 → 1.23, median 2 → 1"
8,H9 vendor leaderboard,supported (Copilot > Claude > OpenAI > Cursor),"S13 2026-04: Copilot 4.25%, Claude 3.83%, Open..."
9,H10 AI mention ≠ ghost job,supported,S14 AI inflated rate 4.5% vs non-AI 5.6%


## 6. Recommended next research moves (ranked)

1. **Big-Tech-stratified RQ1 paper.** The 17pp BT-vs-rest AI-density gap (S10) and the within-firm rise (S17) are strongest at Big Tech; pair with named-firm layoff timelines to test whether AI-density co-moves with announcements.
2. **Vendor-specificity paper (S13).** Nobody has published a labor-demand vendor-share table for dev tools. The hierarchy (Copilot > Claude > OpenAI > Cursor, with ChatGPT plateauing) is entirely new.
3. **YOE-floor inversion paper (S12).** Rule-based YOE in v8 rose compositionally; LLM-YOE declines across all seniority buckets. Contradicts the popular 'scope inflation' narrative. Worth calling out explicitly with the methodological note on rule-vs-LLM YOE.
4. **Within-firm rewrite mechanism (S17) → RQ4 interviews.** v8 established cross-seniority rewriting; S17 localizes it to the within-firm channel. Interview script can ask: 'who inside your firm rewrote the JDs between 2024 and 2026?' Connects employer-side content to the ghost-vs-real debate (RQ4).

## 7. Limitations (v2)

- `unified_core.parquet` is LinkedIn-only by construction — no Indeed sensitivity.
- Control rows in 2024 come from Kaggle (asaniczka + arshkon) matched against Stage-5 `is_control` classification; the 2026 control rows come from explicit scraper query tiers. The *classification criterion* is consistent but the *recruitment surface* differs.
- `unified_core_observations.parquet` has no scrape-cadence for Kaggle sources — S16 posting-survival is 2026 only.
- Vendor-specific regexes match bare words ('cursor', 'claude', 'gemini') — some false positives from unrelated uses (cursor as a UI element, claude as a person's name). Rates should be read directionally.
- Within-firm panel (S17) requires ≥5 SWE postings in both periods; smaller firms are excluded. 292 firms is meaningful but not exhaustive.
- H8 YOE finding depends on `yoe_min_years_llm` coverage (~80% of in-frame SWE rows). Selection bias possible: rows where LLM refused to extract YOE might differ systematically.
- No significance tests. Descriptive only.